# Experiment 01c — fair matched control (full observation)

A confound surfaced in exp01 v1: `matched_classical` inherited the paper's
amputated input (`reuse_indices=[1,2,3]`, cart position discarded), while the
hybrid saw all four observations. Cart position is one of CartPole's two
termination conditions, so the classical arm may have died from **blindness**,
not from being classical. That undercuts the headline "circuit learns where the
equal-budget classical block does not".

This notebook re-runs `matched_classical` with the **full observation**
(`observation="full"`, in_dim 16, width 7, ~135 params) and, as an explicit
ablation, the amputated version too, so the cost of the amputation is measured
rather than assumed.

**Cost:** classical arms, so minutes per cell, not the hours the hybrid took.
The hybrid and the other arms are NOT re-run - their manifests are reused.

---
## 1. Environment (same as always)

In [ ]:
from google.colab import userdata
GITHUB_USER, REPO_NAME, BRANCH = "RogerMas99", "qrl-dissection", "main"
try:
    GH_TOKEN = userdata.get("GH_TOKEN")
    REPO_URL = f"https://{GH_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git" if GH_TOKEN else f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
except Exception:
    REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

import sys, subprocess, pathlib, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    RESULTS = pathlib.Path("/content/drive/MyDrive/tfm_qrl/exp01")
    CODE    = pathlib.Path("/content/qrl-dissection")
else:
    RESULTS = pathlib.Path.cwd() / "results" / "exp01"; CODE = pathlib.Path.cwd()
RESULTS.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    if CODE.exists(): subprocess.run(["git","-C",str(CODE),"pull","--quiet"], check=False)
    else: subprocess.run(["git","clone","--quiet","-b",BRANCH,REPO_URL,str(CODE)], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(CODE/"requirements.txt")], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(CODE)], check=True)
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q","jax","jaxlib"], check=False)

rev = subprocess.run(["git","-C",str(CODE),"rev-parse","--short","HEAD"],capture_output=True,text=True).stdout.strip()
print("code:", CODE, "@", rev, " results:", RESULTS)

import sys as _sys
_need = False
if "autoray.autoray" in _sys.modules:
    import autoray.autoray as _aa; _need = not hasattr(_aa,"NumpyMimic")
if "jax" in _sys.modules: _need = True
if _need:
    print("Incompatible modules loaded -> restarting."); os.kill(os.getpid(), 9)
else:
    print("Clean environment: no restart needed. Continue below.")

### After restarting (if it did), run from here

In [ ]:
import sys, pathlib
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    RESULTS = pathlib.Path("/content/drive/MyDrive/tfm_qrl/exp01")
    CODE    = pathlib.Path("/content/qrl-dissection")
else:
    RESULTS = pathlib.Path.cwd()/"results"/"exp01"; CODE = pathlib.Path.cwd()
sys.path.insert(0, str(CODE/"src"))

import qrl_dissection
from qrl_dissection import build_arm_config, capacity_ladder, analysis
from qrl_dissection.dqn import RunSpec, run_arm, run_grid, GreedyEvalConfig
from qrl_dissection.core.capacity import build_agent_for, count_trainable
print("ready.")

---
## 2. Confirm the fix — full-obs control sees 4 observations, ~135 params

Before running: build both variants and check the input dimension. The fair one
must have `Linear(16, 7)` (16 = 4 obs x 4 repeats); the ablation `Linear(12, 8)`.

In [ ]:
for obs in ("full", "paper"):
    at, cfg = build_arm_config("matched_classical", observation=obs)
    agent = build_agent_for(at, cfg, is_qnet=True)
    n = count_trainable(agent)
    first_linear = [m for m in agent.modules() if m.__class__.__name__ == "Linear"][0]
    print(f"observation={obs:5}  in_features={first_linear.in_features:2}  "
          f"width={cfg['net_arch']}  params={n}  indices={cfg['reuse_indices']}")
print("\nfull  should read 16 in_features (cart position INCLUDED)")
print("paper should read 12 in_features (cart position discarded)")

---
## 3. Run both matched variants

`observation="full"` is the fair control (the one that matters). `observation="paper"`
reproduces v1 for the ablation. Custom run_name via tag so they don't collide
with the v1 manifests already on disk.

In [ ]:
from dataclasses import replace

SEEDS = [1, 2, 3]
STEPS = 60_000
KW = dict(batch_size=128, buffer_size=10_000, train_frequency=10)

def matched_specs(observation, tag):
    # We need build_arm_config to receive observation=..., but RunSpec builds the
    # arm by name. So we register the choice via the tag and a thin wrapper below.
    return [RunSpec(arm="matched_classical", seed=s, fix_autoreset=fix,
                    total_timesteps=STEPS, dqn_kwargs=KW, tag=tag)
            for fix in (False, True) for s in SEEDS]

# run_arm builds the config internally; to pass observation we call build_arm_config
# ourselves and use SafeDQN directly for this experiment.
from qrl_dissection.dqn import SafeDQN
import time, json

def run_matched(observation, tag):
    at, cfg = build_arm_config("matched_classical", observation=observation)
    rows = []
    for fix in (False, True):
        for seed in SEEDS:
            name = f"matched_{tag}__{'fix01on' if fix else 'fix01off'}__s{seed}"
            manifest = RESULTS / f"{name}.manifest.json"
            if manifest.exists():
                print(f"[skip] {name}"); rows.append(json.loads(manifest.read_text())); continue
            print(f"[run ] {name}", flush=True)
            runner = SafeDQN(agent_type=at, agent_config=cfg, run_name=name, seed=seed,
                             fix_autoreset=fix, eval_cfg=GreedyEvalConfig(every_steps=10_000),
                             outdir=RESULTS, **KW)
            out = runner.train(STEPS, progress_bar=False)
            m = {"spec": {"arm": f"matched_{tag}", "seed": seed, "fix_autoreset": fix,
                          "total_timesteps": STEPS},
                 "run_name": name, "outcome": out.__dict__, "agent_config": {k: str(v) for k,v in cfg.items()},
                 "observation": observation, "git_revision": rev}
            manifest.write_text(json.dumps(m, indent=2, default=str))
            rows.append(m)
            p = out.probe
            print(f"       ok  {out.wall_seconds}s  phantoms {100*p['frac_poison']:.2f}%")
    return rows

print("=== FAIR control (full observation) ===")
run_matched("full", "full")
print("\n=== ABLATION (amputated, reproduces v1) — optional, comment out to skip ===")
run_matched("paper", "amp")

---
## 4. The clean comparison

Now `matched_classical (full)` sees the same information as the hybrid. This is
the comparison the whole experiment was built for.

In [ ]:
import pandas as pd, json, numpy as np

def summ(name):
    mp = RESULTS / f"{name}.manifest.json"
    if not mp.exists(): return None
    m = json.loads(mp.read_text())
    csv = m["outcome"]["episodes_csv"]
    rew, step = analysis.load_episodes(csv)
    best = float(pd.Series(rew).rolling(50).mean().max())
    g = m["outcome"].get("eval_csv")
    gb = np.nan
    if g and pathlib.Path(g).exists():
        _, sc = analysis.load_eval(g); gb = float(np.max(sc)) if len(sc) else np.nan
    return dict(name=name, best_ma50=round(best,1), greedy_best=round(gb,1),
                fix01="on" in name, seed=int(name.split("_s")[-1]))

groups = {
    "matched_full":  [f"matched_full__{'fix01on' if f else 'fix01off'}__s{s}" for f in (False,True) for s in (1,2,3)],
    "matched_amp":   [f"matched_amp__{'fix01on' if f else 'fix01off'}__s{s}" for f in (False,True) for s in (1,2,3)],
    "hybrid_fig4":   [f"hybrid_fig4__{'fix01on' if f else 'fix01off'}__s{s}" for f in (False,True) for s in (1,2,3)],
}
rows = []
for arm, names in groups.items():
    for nm in names:
        r = summ(nm)
        if r: r["arm"] = arm; rows.append(r)
df = pd.DataFrame(rows)
if len(df):
    piv = df.groupby("arm").agg(best_mean=("best_ma50","mean"), best_std=("best_ma50","std"),
                                greedy_mean=("greedy_best","mean")).round(1)
    display(piv)
    print("\n=== headline: does the circuit still win at equal budget AND equal information? ===")
    for a in ("matched_full","hybrid_fig4"):
        if a in piv.index:
            print(f"  {a:16} best_ma50 {piv.loc[a,'best_mean']:6.1f}  greedy {piv.loc[a,'greedy_mean']:6.1f}")
    if "matched_amp" in piv.index and "matched_full" in piv.index:
        d = piv.loc['matched_full','best_mean'] - piv.loc['matched_amp','best_mean']
        print(f"\n  cost of the amputation (full - amputated): {d:+.1f} best_ma50")

---
## 5. Reading it

Three arms now sit at ~130 params: `matched_amp` (blind), `matched_full`
(sighted), `hybrid_fig4`. Compare in that order.

**matched_full still dies (~23), hybrid learns (~54):** the headline holds and is
now clean — at equal budget AND equal information, the circuit learns where the
classical block collapses. The amputation was not the cause. This is the strong
result.

**matched_full now learns:** then v1's death was largely the missing cart
position, not the absence of a circuit. Compare its best_ma50 to the hybrid's 54:
- comparable -> no clear circuit advantage under DQN; the honest conclusion is
  that the paper's PPO finding does not transfer, and the earlier "circuit wins"
  reading was a confound. That is still a solid, publishable result — a negative
  one, honestly obtained.
- still well below 54 -> a weaker circuit advantage survives.

**The amputation cost (full - amp)** quantifies, as a bonus, how much hiding a
termination variable hurts a linear-ish agent — a clean little ablation.

Whatever happens: update `docs/RESULTS-LOG.md` (exp01 v2 table), commit, and
re-tag. The v1 numbers stay in the log labelled as the amputation ablation.